# Técnica de Pooling con SVM en entornos aislados y señas continuas

# Librerías

In [ ]:
from __future__ import annotations

import json
import os
import random
import re
from dataclasses import dataclass
import pathlib
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import csv
from datetime import datetime
from collections import Counter


import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

import mediapipe as mp        
import joblib                 # guardar/cargar modelos
import matplotlib.pyplot as plt
import seaborn as sns       

# ---------- Scikit-learn ----------
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
)


from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    ParameterGrid,
    StratifiedKFold,
    train_test_split,
    cross_validate,
)

try:
    from sklearn.model_selection import StratifiedGroupKFold  
except Exception:
    StratifiedGroupKFold = None

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.linear_model import LinearRegression
from sklearn.inspection import permutation_importance

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score, accuracy_score, ConfusionMatrixDisplay

In [ ]:
DEBUG = True
TMP_DEBUG = True
CHANGE_DIR_NAMES = False # Se cambia a True cuando ya se verifica
CREATE_MODELS = True
USE_CSV_KEYPOINTS = True  # Si es True, se cargan los CSV ya generados
T = 30                       # frames
RANDOM_STATE = 42
SCORING = "f1_macro"


ROOTS = [
    Path(r"DB_LSM\MSL-dynamic-signs\train"),
    Path(r"DB_LSM\MSL-dynamic-signs\test")
    
]

KEYPOINT_PATH = Path(r"Dinamicos_Aislado\Features\keypoints_30f.csv")
ROOT_TRAIN = Path(r"DB_LSM\MSL-dynamic-signs\train")
ROOT_TEST = Path(r"DB_LSM\MSL-dynamic-signs\test")

mp_hands = mp.solutions.hands

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

### Comprobación de uso de los keypoints ya creados

In [ ]:
df = pd.read_csv(KEYPOINT_PATH)

print(df.shape)
print(df.head(3))

### Expresión regular

In [ ]:
ER = re.compile(r"[\\/](S\d+)-([A-Za-zÑñ])(?:[-_])", flags=re.IGNORECASE)

### Agrupación de frames por keypoint

In [ ]:
grouped = df.groupby("path")

video_path, video_frames = next(iter(grouped))

if DEBUG:
    first5 = list(grouped)[:5]
    for path, g in first5:
        print("\nVideo:", path)
        print("Features por frame:", g.filter(like="f").shape[1])


### Función de busqueda de clase (Letra)

In [ ]:
def get_class_from_path(p: str) -> str:
    m = ER.search(str(p))
    return m.group(2).upper() if m else "UNK"

In [ ]:
df["label"] = df["path"].astype(str).apply(get_class_from_path)

#### Comprobación de labels conflictivas

In [ ]:
bad = [lab for lab in df["label"].unique() if lab.lower() in ("train","test") or lab == "UNK"]
print("🔎 labels conflictivas:", bad)

In [ ]:
unk_paths = df.loc[df["label"]=="UNK", "path"].head(10).tolist()
print("Ejemplos UNK:", unk_paths)
print("Total UNK:", (df["label"]=="UNK").sum())

Si hay clases que no se pueden conseguir correctamente, entonces aquí saldrán

### Función de creación de vectores de pooling basico

In [ ]:
def pooling_from_group(group_df, T : int):

    X_seq = group_df.filter(like="f").values.astype(np.float32)

    if X_seq.shape[0] >= T:
        X_seq = X_seq[:T, :]
    else:
        pad = np.full((T - X_seq.shape[0], X_seq.shape[1]), np.nan, dtype=np.float32)
        X_seq = np.vstack([X_seq, pad])

    mean = np.nanmean(X_seq, axis=0)
    std  = np.nanstd(X_seq, axis=0, ddof=0)

    return np.concatenate([mean, std], axis=0)

Esta es una de las funciones principales del codigo, en donde se sacan las caracteristicas estadisticas en base de las posiciones de los keypoints. En este caso, siendo una concatenacion de (media, desviacion_estandar).

### Función de creación de vectores de poolingXL

In [ ]:
def pooling_from_groupXL(group_df, T : int):

    X_seq = group_df.filter(like="f").values.astype(np.float32)

    if X_seq.shape[0] >= T:
        X_seq = X_seq[:T, :]
    else:
        pad = np.full((T - X_seq.shape[0], X_seq.shape[1]), np.nan, dtype=np.float32)
        X_seq = np.vstack([X_seq, pad])

    mean   = np.nanmean(X_seq, axis=0)      
    std    = np.nanstd(X_seq, axis=0, ddof=0) 
    median = np.nanmedian(X_seq, axis=0)    
    p10    = np.nanpercentile(X_seq, 10, axis=0)  # percentil 10
    p90    = np.nanpercentile(X_seq, 90, axis=0)  # percentil 90

    return np.concatenate([mean, std, median, p10, p90], axis=0)


Con esta funcion podremos hacer una prueba mas extensa, al tener mas features, siendo en este caso (media, desviacion estandar, mediana, percentil_10, percentil_90).

### Preparación de Diccionario con DF para entrenamiento de modelo

In [ ]:
df["label"] = df["path"].apply(get_class_from_path)

print(df[["path", "label"]].head(10))

#### Agrupacion Pooling Basico

In [ ]:
X_pool_list = []
y_list = []
paths_list = []

for path, g in grouped:
    pooled = pooling_from_group(g, T)
    X_pool_list.append(pooled)
    y_list.append(g["label"].iloc[0])  
    paths_list.append(path)

X_pool = np.vstack(X_pool_list)
y_pool = np.array(y_list)
paths = np.array(paths_list)

if DEBUG:
    print("X_pool:", X_pool.shape)  
    print("y_pool:", y_pool.shape)  
    print("Ejemplo path:", paths[0])

#### Agrupacion Pooling XL

In [ ]:
X_pool_listXL = []
y_listXL = []
paths_listXL = []

for pathXL, gXL in grouped:
    pooledXL = pooling_from_groupXL(gXL, T)
    X_pool_listXL.append(pooledXL)
    y_listXL.append(gXL["label"].iloc[0])  
    paths_listXL.append(pathXL)

X_poolXL = np.vstack(X_pool_listXL)
y_poolXL = np.array(y_listXL)
pathsXL = np.array(paths_listXL)

if DEBUG:
    print("X_poolXL:", X_poolXL.shape)  
    print("y_poolXL:", y_poolXL.shape)  
    print("Ejemplo pathXL:", pathsXL[0])


In [ ]:
assert np.all(y_pool == y_poolXL), "Cuidado: las etiquetas no coinciden!"
assert np.all(paths == pathsXL), "Cuidado: los paths no coinciden!"

Como coincide (O deberia), quiere decir que es arbitrario el uso de cualquier factor y que utilicemos, por ende, utilizaremos solo el del y_pool normal.

In [ ]:
feature_sets = {
    "Pool_Basic":  X_pool,          
     "Pool_XL":   X_poolXL
}

print("Feature sets:", list(feature_sets.keys()))
for k, X in feature_sets.items():
    print(k, getattr(X, "shape", None))

feature_sets = {k:v for k,v in feature_sets.items() if v is not None}

le = LabelEncoder()
y_enc = le.fit_transform(y_pool)
classes = le.classes_
print("Clases:", list(classes))

### Función de Evaluación de SVM

In [ ]:
def eval_svm(name, X, y_enc, is_train, is_test, classes, outdir=OUTDIR):
    print(f"\n=== {name} ===")
    X_tr, y_tr = X[is_train], y_enc[is_train]
    X_te, y_te = X[is_test],  y_enc[is_test]
    print("Train:", X_tr.shape, "| Test:", X_te.shape)

    pipe = make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)
    )
    param_grid = {
        "svc__C":     [1, 3, 10, 30, 100],
        "svc__gamma": ["scale", 0.03, 0.1, 0.3],
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    gscv = GridSearchCV(
        pipe, param_grid,
        scoring=SCORING, refit=SCORING,
        cv=cv, n_jobs=-1, verbose=1
    )

    gscv.fit(X_tr, y_tr)
    print("Mejores params:", gscv.best_params_)
    print(f"CV {SCORING}:", gscv.best_score_)

    best_model = gscv.best_estimator_
    y_pred = best_model.predict(X_te)

    acc = accuracy_score(y_te, y_pred)
    f1m = f1_score(y_te, y_pred, average="macro")
    print("Test Acc:", acc)
    print("Test F1-macro:", f1m)

    rep = classification_report(y_te, y_pred, target_names=classes, output_dict=True)
    df_rep = pd.DataFrame(rep).transpose()
    df_rep.to_csv(outdir / f"report_{name}.csv")

    plt.figure(figsize=(6,5))
    ConfusionMatrixDisplay.from_predictions(
        y_te, y_pred,
        display_labels=classes,  
        cmap="viridis",
        values_format="d"
    )
    plt.title(f"Confusion Matrix — {name}")
    plt.tight_layout()
    plt.show()

    joblib.dump(best_model, outdir / f"svm_{name}_best.joblib")
    joblib.dump(le,         outdir / f"label_encoder.joblib")

    return {
        "name": name,
        "cv_f1_macro": gscv.best_score_,
        "test_acc": acc,
        "test_f1_macro": f1m,
        "n_features": X.shape[1]
    }


Esta funcion realizara toda la evaluacion de los modelos SVM.
Hara un GridSearch con respecto a los parametros **C** y **gamma**.
Lo evalua con un **CV** de **kFold = 5** 

=== HACER CON EL LEAVE ONE GROUP OUT ===


Crea un reporte de 
Guardara el modelo con la libreria **joblib**

### Evaluación de diccionarios con features de los keypoints por SVM

#### Evaluación de pooling y poolingXL

In [ ]:
results = []
for name, X in feature_sets.items():
    results.append(eval_svm(name, X, y_enc, is_train, is_test, classes))

df_res = pd.DataFrame(results).sort_values("test_f1_macro", ascending=False)
display(df_res)
df_res.to_csv(OUTDIR / "summary_results.csv", index=False)
print("✅ Guardados reportes y modelos en:", OUTDIR.resolve())

Salió exactamente el mismo resultado, esto quiere decir que o hay un problema de leakage, se reutilizaron variables que no debían o las features que propone el pooling XL, son redundantes, entonces realizaremos una comprobación para ver qué tan importantes son las features del DF.

#### Comprobación de importancia de Features

In [ ]:
X_basic_from_XL = X_poolXL[:, :126]
X_extra         = X_poolXL[:, 126:]

var_extra = X_extra.var(axis=0)
print("Casi constantes:", (var_extra < 1e-10).sum(), "de", X_extra.shape[1])

#(R2 alto = redundante)
lr = LinearRegression().fit(X_basic_from_XL, X_extra)
r2 = lr.score(X_basic_from_XL, X_extra)
print("R2(extra ~ basic):", r2)

### Permutation Importance

Ya que vimos que muchas de las features extras del pooling XL son una transformación lineal del basico, es decir, no aportan mucho y quedan redundantes al no proveer de nueva información al modelo, entonces veremos qué tanto del pooling básico necesitamos y qué podemos desechar.

#### Función de recolección de train y test

In [ ]:
def split_data(X, y_enc, is_train, is_test):
    X_tr, y_tr = X[is_train], y_enc[is_train]
    X_te, y_te = X[is_test],  y_enc[is_test]
    return X_tr, X_te, y_tr, y_te

#### Configuración para diccionario de poolingXL

In [ ]:
feature_sets = {
    "Pool_XL": X_poolXL
}

print("Feature sets:", list(feature_sets.keys()))
for k, X in feature_sets.items():
    print(k, getattr(X, "shape", None))

feature_sets = {k: v for k, v in feature_sets.items() if v is not None}

le = LabelEncoder()
y_enc = le.fit_transform(y_pool)
classes = le.classes_
print("Clases:", list(classes))

#### Separación de Train y Test en PoolingXL

In [ ]:
X = feature_sets["Pool_XL"]
X_tr, X_te, y_tr, y_te = split_data(X, y_enc, is_train, is_test)
print("Train:", X_tr.shape, "| Test:", X_te.shape)

#### Carga de modelo

In [ ]:
modelXL = joblib.load(r"Dinamicos_Aislado\Pooling\svm_Pool_XL_best.joblib")

#### Aplicación de Permutation Importance

In [ ]:
res = permutation_importance(
    modelXL,
    X_te, y_te,
    n_repeats=20,
    random_state=RANDOM_STATE,
    scoring="f1_macro"
)

imp_mean = res.importances_mean
imp_std  = res.importances_std
idx_ord  = np.argsort(imp_mean)[::-1]

#### Creación de DF para sencillo manejo

In [ ]:
df_pi = pd.DataFrame({
    "feature_idx": np.arange(X.shape[1]),
    "pi_mean": imp_mean,
    "pi_std": imp_std
}).iloc[idx_ord].reset_index(drop=True)
df_pi["rank"] = np.arange(1, len(df_pi) + 1)

#### Configuración para renombrado de features

In [ ]:
STATS = ["mean", "std", "median", "p10", "p90"]
ORDER = "stats_first"

D = X_poolXL.shape[1] // len(STATS)

try:
    base_cols = list(df.filter(like="f").columns)
    assert len(base_cols) == D
except Exception:
    # Fallback: nombres genéricos f00, f01, ...
    base_cols = [f"f{j:02d}" for j in range(D)]

#### Función de renombrado de feature a nombre real

In [ ]:
def idx_to_name(idx:int):
    s = idx // D         # qué estadístico
    b = idx %  D         # qué feature dentro del bloque
    stat = STATS[s]
    base = base_cols[b]

    return f"{base}|{stat}", base, stat

#### Aplicación de renombrado

In [ ]:
df_names = df_pi.copy()
df_names[["feature_name","base_feature","stat"]] = df_names["feature_idx"] \
    .apply(lambda i: pd.Series(idx_to_name(int(i))))

df_names.to_csv(OUTDIR / "PI_Pool_XL_with_names.csv", index=False)
TOP_N = 25
top = df_names.head(TOP_N).copy()

### Gráfico de Feature(Permutation) Importance

In [ ]:

plt.figure(figsize=(14,6))
ax = sns.barplot(data=top, x="feature_name", y="pi_mean", dodge=False, ci=None)
# barras de error (std de la permutación)
ax.errorbar(
    x=np.arange(len(top)),
    y=top["pi_mean"].values,
    yerr=top["pi_std"].values,
    fmt="none", ecolor="black", capsize=3, linewidth=1
)
ax.set_title(f"Top {TOP_N} features — Pool_XL (Permutation Importance, ΔF1)")
ax.set_xlabel("Feature (base|stat)")
ax.set_ylabel("Importancia media (ΔF1)")
plt.xticks(rotation=70, ha="right")
plt.tight_layout()
plt.show()

print(top[["rank","feature_idx","feature_name","pi_mean","pi_std"]])

### Entrenamiento con Top-K y N features

In [ ]:
PARAM_GRID = {
    "svc__C":     [1, 3, 10, 30, 100],
    "svc__gamma": ["scale", 0.03, 0.1, 0.3],
}

def build_svm():
    return make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)
    )

feat_order = df_pi["feature_idx"].to_numpy()

feat_order = df_pi.sort_values("pi_mean", ascending=False)["feature_idx"].to_numpy()

print("Orden de features (top 10):", feat_order[:10])

#### Función de Rankeo con Cross Validation

In [ ]:
def pi_ranking_cv(X_tr, y_tr, cv_splits=5, n_repeats=10):
    skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    nfeat = X_tr.shape[1]
    agg_importances = np.zeros(nfeat, dtype=float)
    counts = np.zeros(nfeat, dtype=float)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr, y_tr), 1):
        X_tr_in, X_va_in = X_tr[tr_idx], X_tr[va_idx]
        y_tr_in, y_va_in = y_tr[tr_idx], y_tr[va_idx]

        pipe = build_svm()
        gscv = GridSearchCV(pipe, PARAM_GRID, scoring=SCORING, cv=5, n_jobs=-1, refit=True, verbose=0)
        gscv.fit(X_tr_in, y_tr_in)

        best = gscv.best_estimator_
        res = permutation_importance(best, X_va_in, y_va_in,
                                     n_repeats=n_repeats, random_state=RANDOM_STATE,
                                     scoring=SCORING)
        agg_importances += res.importances_mean
        counts += 1

    mean_imp = agg_importances / np.maximum(counts, 1)
    order = np.argsort(mean_imp)[::-1]
    return order, mean_imp

### Top-K

#### Funciones

In [ ]:
def train_top_k(X, y, is_train, is_test, feat_order, K,
                scoring=SCORING, return_pred=False, save_path=None):
    
    K = int(min(K, X.shape[1]))
    sel = np.array(feat_order[:K], dtype=int)

    X_tr, y_tr = X[is_train][:, sel], y[is_train]
    X_te, y_te = X[is_test][:,  sel], y[is_test]

    gs = GridSearchCV(build_svm(), PARAM_GRID, scoring=scoring, cv=5,
                      n_jobs=-1, refit=True, verbose=0)
    gs.fit(X_tr, y_tr)
    model = gs.best_estimator_

    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    f1m = f1_score(y_te, y_pred, average="macro")

    out = {
        "K": K,
        "sel_idx": sel,
        "best_params": gs.best_params_,
        "test_acc": acc,
        "test_f1_macro": f1m,
        "model": model,
    }
    if return_pred:
        out.update({"y_true": y_te, "y_pred": y_pred})

    if save_path is not None:
        joblib.dump(model, save_path)

    return out


In [ ]:
def run_topk_sweep(X, y, is_train, is_test, feat_order, Ks, save_prefix=None):

    rows = []
    for K in Ks:
        res = train_top_k(X, y, is_train, is_test, feat_order, K, scoring=SCORING)
        rows.append({
            "K": int(res["K"]),
            "F1": float(res["test_f1_macro"]),
            "Acc": float(res["test_acc"]),
            "C": res["best_params"]["svc__C"],
            "gamma": res["best_params"]["svc__gamma"],
        })
        print(f"K={K:>4}  F1={rows[-1]['F1']:.4f}  Acc={rows[-1]['Acc']:.4f}  "
              f"best(C={rows[-1]['C']}, gamma={rows[-1]['gamma']})")

    df = pd.DataFrame(rows).sort_values("K").reset_index(drop=True)

    if save_prefix:
        csv_path = OUTDIR / f"{save_prefix}_topk_sweep.csv"
        df.to_csv(csv_path, index=False)
        print("CSV guardado en:", csv_path)

    return df

In [ ]:
def plot_topk(df, title="Top-K sweep", save_prefix=None):

    df_long = df.melt(id_vars=["K"], value_vars=["F1", "Acc"],
                      var_name="métrica", value_name="score")

    plt.figure(figsize=(9,5))
    ax = sns.lineplot(data=df_long, x="K", y="score", hue="métrica", marker="o")

    i_best = df["F1"].idxmax()
    bestK = int(df.loc[i_best, "K"])
    bestF1 = float(df.loc[i_best, "F1"])

    ax.axvline(bestK, ls="--", alpha=0.35)
    ax.scatter([bestK], [bestF1], s=80, zorder=5)
    ax.annotate(f"best K={bestK}\nF1={bestF1:.3f}",
                xy=(bestK, bestF1), xytext=(12,12), textcoords="offset points",
                fontsize=9, bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8))

    ax.set_ylim(0, 1)
    ax.set_xlabel("Top-K features")
    ax.set_ylabel("Score")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    plt.tight_layout()

    if save_prefix:
        png_path = OUTDIR / f"{save_prefix}_topk_sweep.png"
        plt.savefig(png_path, dpi=200, bbox_inches="tight")
        print("Figura guardada en:", png_path)

    plt.show()

#### Aplicación de TrainTopK

In [ ]:
step = 15
Ks = list(range(step, X.shape[1] + 1, step))
if Ks[-1] != X.shape[1]:
    Ks.append(X.shape[1])

df_topk = run_topk_sweep(X_poolXL, y_enc, is_train, is_test, feat_order, Ks, save_prefix="Pool_XL")
plot_topk(df_topk, title="Pool_XL — Top-K sweep", save_prefix="Pool_XL")
